# Classic ML Test Majority Vote Metrics (one model per ESP)

Notebook повторяет train/test pipeline из `classic_ml_train.ipynb`, но обучает **отдельную модель для каждой ESP-платы** (`dev1`, `dev2`, `dev3`).

Для каждой ESP выполняется свой train/test split по папкам `test_N`, считаются свои preprocessing-объекты и сохраняются отдельные артефакты.

Метрики считаются только на test split, отдельно по каждой плате.

In [1]:
from pathlib import Path
import json
import re
import sys

import joblib
import numpy as np
import pandas as pd

from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, precision_recall_fscore_support
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

project_root = Path.cwd()
if not (project_root / 'tools').exists() and (project_root.parent / 'tools').exists():
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from tools.csi_parser import Parser
from tools.filters import median_filter
from tools.classic_ml_inference import run_inference

dataset_root = project_root / 'wifi_data_set_fixed_split'
median_width = 5
random_state = 42

if median_width % 2 == 0:
    raise ValueError('median_width must be odd')

np.random.seed(random_state)

print(f'project_root: {project_root}')
print(f'dataset_root: {dataset_root}')

project_root: /home/gleb/learning/CSI-activity-detection
dataset_root: /home/gleb/learning/CSI-activity-detection/wifi_data_set_fixed_split


## Helpers

Эти функции повторяют preprocessing из `classic_ml_train.ipynb` и добавляют извлечение `esp_id` из имени файла (`...dev1...`, `...dev2...`, `...dev3...`).

In [2]:
def extract_esp_id(file_path: Path) -> str:
    stem = file_path.stem
    match = re.search(r'dev(\d+)', stem, flags=re.IGNORECASE)
    if not match:
        raise ValueError(f'Could not extract ESP id from file name: {file_path.name}')
    return f'dev{int(match.group(1))}'


def parse_path_meta(file_path: Path) -> dict:
    parts = file_path.parts
    return {
        'person_id': next((p for p in parts if p.startswith('id_person_')), 'unknown'),
        'label': next((p for p in parts if p.startswith('label_')), 'unknown'),
        'test_id': next((p for p in parts if p.startswith('test_')), 'unknown'),
        'esp_id': extract_esp_id(file_path) if file_path.suffix == '.data' else 'unknown',
    }


def get_test_group_dir(file_path: Path) -> Path:
    for parent in file_path.parents:
        if parent.name.startswith('test_'):
            return parent
    raise ValueError(f'Could not find test_* folder in path: {file_path}')


def skewness(x: np.ndarray) -> float:
    x = np.asarray(x, dtype=np.float64)
    mu = x.mean()
    sigma = x.std()
    if sigma == 0:
        return 0.0
    z = (x - mu) / sigma
    return float(np.mean(z ** 3))


def kurtosis_excess(x: np.ndarray) -> float:
    x = np.asarray(x, dtype=np.float64)
    mu = x.mean()
    sigma = x.std()
    if sigma == 0:
        return 0.0
    z = (x - mu) / sigma
    return float(np.mean(z ** 4) - 3.0)


def spectral_features(signal: np.ndarray) -> tuple[float, float, float, float, float]:
    x = np.asarray(signal, dtype=np.float64)
    n = len(x)
    if n < 2:
        return 0.0, 0.0, 0.0, 0.0, 0.0

    spectrum = np.fft.rfft(x)
    power = np.abs(spectrum) ** 2
    freqs = np.fft.rfftfreq(n, d=1.0)

    power_sum = power.sum()
    if power_sum <= 0:
        return 0.0, 0.0, 0.0, 0.0, 0.0

    centroid = float((freqs * power).sum() / power_sum)
    spread = float(np.sqrt(((freqs - centroid) ** 2 * power).sum() / power_sum))
    p_norm = power / power_sum
    entropy = float(-(p_norm * np.log2(p_norm + 1e-12)).sum())
    dominant_freq = float(freqs[int(np.argmax(power))])
    rolloff = float(freqs[np.searchsorted(np.cumsum(power), 0.85 * power_sum)])

    return centroid, spread, entropy, dominant_freq, rolloff


def build_unit_signal(df: pd.DataFrame, window: int) -> np.ndarray:
    amp_matrix = np.stack(df['amplitude'].to_numpy(), axis=0).astype(np.float64)
    raw_time_signal = amp_matrix.mean(axis=1)
    return median_filter(raw_time_signal, window)


def build_unit_cache(file_list: list[Path]) -> list[dict]:
    cache = []
    for file_path in file_list:
        df = Parser(file_path).parse()
        filtered_signal = build_unit_signal(df, median_width)
        cache.append({
            'file_path': file_path,
            'df': df,
            'filtered_signal': filtered_signal,
        })
    return cache

## Group train/test split per ESP

Split делается по папкам `test_N`, но **отдельно для каждой ESP**: у каждой платы свой train/test набор групп.

In [3]:
all_files = sorted(dataset_root.rglob('*.data'))
if not all_files:
    raise FileNotFoundError(f'No .data files found under {dataset_root}')

file_rows = []
for fp in all_files:
    meta = parse_path_meta(fp)
    file_rows.append({
        'file_path': fp,
        'group_path': get_test_group_dir(fp),
        'label': meta['label'],
        'esp_id': meta['esp_id'],
    })

files_df = pd.DataFrame(file_rows)
esp_ids = sorted(files_df['esp_id'].unique())

esp_splits: dict[str, dict] = {}
for esp_id in esp_ids:
    esp_df = files_df[files_df['esp_id'] == esp_id].copy()
    esp_groups = sorted(esp_df['group_path'].unique())

    group_labels = [parse_path_meta(group_path)['label'] for group_path in esp_groups]

    train_groups, test_groups = train_test_split(
        esp_groups,
        test_size=0.2,
        random_state=random_state,
        stratify=group_labels,
        shuffle=True,
    )

    train_group_set = set(train_groups)
    test_group_set = set(test_groups)

    shared_groups = train_group_set & test_group_set
    if shared_groups:
        raise RuntimeError(f'Group leakage detected for {esp_id}: {sorted(shared_groups)}')

    train_files = sorted([fp for fp in esp_df['file_path'] if get_test_group_dir(fp) in train_group_set])
    test_files = sorted([fp for fp in esp_df['file_path'] if get_test_group_dir(fp) in test_group_set])

    esp_splits[esp_id] = {
        'train_groups': train_groups,
        'test_groups': test_groups,
        'train_files': train_files,
        'test_files': test_files,
    }

print(f'Total .data files: {len(all_files)}')
print(f'ESP ids found: {esp_ids}')
for esp_id in esp_ids:
    split = esp_splits[esp_id]
    print(
        f"{esp_id}: train_groups={len(split['train_groups'])}, "
        f"test_groups={len(split['test_groups'])}, "
        f"train_files={len(split['train_files'])}, test_files={len(split['test_files'])}"
    )

Total .data files: 24000
ESP ids found: ['dev1', 'dev2', 'dev3']
dev1: train_groups=6400, test_groups=1600, train_files=6400, test_files=1600
dev2: train_groups=6400, test_groups=1600, train_files=6400, test_files=1600
dev3: train_groups=6400, test_groups=1600, train_files=6400, test_files=1600


## Feature extraction per ESP

Global amplitude min/max считаются только на train и отдельно для каждой ESP.

In [4]:
def build_features_df(unit_cache: list[dict], global_min: float, global_max: float) -> pd.DataFrame:
    den = global_max - global_min
    if den <= 0:
        raise ValueError('Global max equals global min; cannot perform min-max scaling')

    rows = []
    for unit in unit_cache:
        file_path = unit['file_path']
        df = unit['df']
        signal = unit['filtered_signal']

        scaled_signal = (signal - global_min) / den
        scaled_signal = np.clip(scaled_signal, 0.0, 1.0)

        mean_val = float(np.mean(scaled_signal))
        std_val = float(np.std(scaled_signal))
        median_val = float(np.median(scaled_signal))
        min_val = float(np.min(scaled_signal))
        max_val = float(np.max(scaled_signal))
        q25 = float(np.percentile(scaled_signal, 25))
        q75 = float(np.percentile(scaled_signal, 75))
        iqr = q75 - q25
        range_val = max_val - min_val
        rms = float(np.sqrt(np.mean(scaled_signal ** 2)))
        energy = float(np.sum(scaled_signal ** 2))
        zcr = float(np.mean(np.abs(np.diff(np.signbit(scaled_signal - mean_val)).astype(np.int8))))

        sk = skewness(scaled_signal)
        kt = kurtosis_excess(scaled_signal)
        sc, ss, se, dom_freq, rolloff = spectral_features(scaled_signal)
        meta = parse_path_meta(file_path)

        rows.append({
            'file_path': str(file_path),
            'group_path': str(get_test_group_dir(file_path)),
            **meta,
            'n_packets': int(len(df)),
            'mean': mean_val,
            'std': std_val,
            'median': median_val,
            'skew': sk,
            'kurtosis': kt,
            'spectral_centroid': sc,
            'spectral_spread': ss,
            'spectral_entropy': se,
            'dominant_freq': dom_freq,
            'spectral_rolloff_85': rolloff,
            'min': min_val,
            'max': max_val,
            'q25': q25,
            'q75': q75,
            'iqr': iqr,
            'range': range_val,
            'rms': rms,
            'energy': energy,
            'zcr': zcr,
        })

    return pd.DataFrame(rows)


feature_cols = [
    'mean', 'std', 'median', 'skew', 'kurtosis',
    'spectral_centroid', 'spectral_spread', 'spectral_entropy',
    'dominant_freq', 'spectral_rolloff_85',
    'min', 'max', 'q25', 'q75', 'iqr', 'range',
    'rms', 'energy', 'zcr',
]

esp_artifacts: dict[str, dict] = {}
for esp_id in esp_ids:
    split = esp_splits[esp_id]
    train_unit_cache = build_unit_cache(split['train_files'])
    test_unit_cache = build_unit_cache(split['test_files'])

    global_min = min(float(np.min(unit['filtered_signal'])) for unit in train_unit_cache)
    global_max = max(float(np.max(unit['filtered_signal'])) for unit in train_unit_cache)

    train_features_df = build_features_df(train_unit_cache, global_min, global_max)
    test_features_df = build_features_df(test_unit_cache, global_min, global_max)

    esp_artifacts[esp_id] = {
        'global_min': global_min,
        'global_max': global_max,
        'train_features_df': train_features_df,
        'test_features_df': test_features_df,
    }

    print(
        f"{esp_id}: train_features={train_features_df.shape}, "
        f"test_features={test_features_df.shape}, "
        f"global_min={global_min:.6f}, global_max={global_max:.6f}"
    )

dev1: train_features=(6400, 26), test_features=(1600, 26), global_min=1.184494, global_max=29.857592
dev2: train_features=(6400, 26), test_features=(1600, 26), global_min=3.374140, global_max=39.403599
dev3: train_features=(6400, 26), test_features=(1600, 26), global_min=0.494638, global_max=14.640834


## PCA preprocessing per ESP

`StandardScaler` и `PCA` обучаются только на train и отдельно для каждой ESP.

In [5]:
for esp_id in esp_ids:
    train_features_df = esp_artifacts[esp_id]['train_features_df']
    test_features_df = esp_artifacts[esp_id]['test_features_df']

    X_train_raw = train_features_df[feature_cols].to_numpy(dtype=np.float64)
    X_test_raw = test_features_df[feature_cols].to_numpy(dtype=np.float64)

    feature_min = X_train_raw.min(axis=0)
    feature_max = X_train_raw.max(axis=0)

    X_train = np.clip(X_train_raw, feature_min, feature_max)
    X_test = np.clip(X_test_raw, feature_min, feature_max)

    scaler_pca = StandardScaler()
    X_train_std = scaler_pca.fit_transform(X_train)
    X_test_std = scaler_pca.transform(X_test)

    pca = PCA()
    X_train_pca = pca.fit_transform(X_train_std)
    X_test_pca = pca.transform(X_test_std)

    cum_evr = np.cumsum(pca.explained_variance_ratio_)
    k_95 = int(np.searchsorted(cum_evr, 0.95) + 1)

    X_train_pca_95 = X_train_pca[:, :k_95]
    X_test_pca_95 = X_test_pca[:, :k_95]

    y_train = (train_features_df['label'] != 'label_00').astype(int).to_numpy()
    y_test_file = (test_features_df['label'] != 'label_00').astype(int).to_numpy()

    esp_artifacts[esp_id].update({
        'feature_min': feature_min,
        'feature_max': feature_max,
        'scaler_pca': scaler_pca,
        'pca': pca,
        'k_95': int(k_95),
        'X_train_pca_95': X_train_pca_95,
        'X_test_pca_95': X_test_pca_95,
        'y_train': y_train,
        'y_test_file': y_test_file,
    })

    print(
        f"{esp_id}: k_95={k_95}, X_train_pca_95={X_train_pca_95.shape}, "
        f"X_test_pca_95={X_test_pca_95.shape}, train_class_1={(y_train == 1).sum()}, "
        f"test_class_1={(y_test_file == 1).sum()}"
    )

dev1: k_95=5, X_train_pca_95=(6400, 5), X_test_pca_95=(1600, 5), train_class_1=4800, test_class_1=1200
dev2: k_95=6, X_train_pca_95=(6400, 6), X_test_pca_95=(1600, 6), train_class_1=4800, test_class_1=1200
dev3: k_95=6, X_train_pca_95=(6400, 6), X_test_pca_95=(1600, 6), train_class_1=4800, test_class_1=1200


## Train one best model per ESP

Для каждой ESP обучается отдельная модель: `SVM(C=5.0, gamma='auto', kernel='rbf')`.

In [6]:
for esp_id in esp_ids:
    model = SVC(C=5.0, gamma='auto', kernel='rbf', probability=True, random_state=random_state)
    model.fit(esp_artifacts[esp_id]['X_train_pca_95'], esp_artifacts[esp_id]['y_train'])

    file_pred = model.predict(esp_artifacts[esp_id]['X_test_pca_95'])
    y_test_file = esp_artifacts[esp_id]['y_test_file']

    esp_artifacts[esp_id].update({
        'model': model,
        'file_pred': file_pred,
    })

    print(f'[{esp_id}] File-level test metrics before majority vote:')
    print(f'accuracy: {accuracy_score(y_test_file, file_pred):.4f}')
    print(classification_report(
        y_test_file,
        file_pred,
        labels=[0, 1],
        target_names=['static / no motion', 'dynamic / motion'],
        zero_division=0,
    ))

[dev1] File-level test metrics before majority vote:
accuracy: 0.8969
                    precision    recall  f1-score   support

static / no motion       0.77      0.83      0.80       400
  dynamic / motion       0.94      0.92      0.93      1200

          accuracy                           0.90      1600
         macro avg       0.86      0.88      0.87      1600
      weighted avg       0.90      0.90      0.90      1600

[dev2] File-level test metrics before majority vote:
accuracy: 0.8206
                    precision    recall  f1-score   support

static / no motion       0.71      0.48      0.57       400
  dynamic / motion       0.84      0.94      0.89      1200

          accuracy                           0.82      1600
         macro avg       0.78      0.71      0.73      1600
      weighted avg       0.81      0.82      0.81      1600

[dev3] File-level test metrics before majority vote:
accuracy: 0.9313
                    precision    recall  f1-score   support

sta

## Save notebook artifacts per ESP

`run_inference` из `tools/classic_ml_inference.py` используется для каждой ESP-модели отдельно, поэтому сохраняются отдельные `model/preprocessing/metadata` в подпапки `esp_*`.

In [7]:
artifacts_root = project_root / 'artifacts' / 'classic_ml_majority_vote_metrics'
artifacts_root.mkdir(parents=True, exist_ok=True)

for esp_id in esp_ids:
    esp_dir = artifacts_root / esp_id
    esp_dir.mkdir(parents=True, exist_ok=True)

    model_path = esp_dir / 'svm_model.joblib'
    preprocessing_path = esp_dir / 'preprocessing.joblib'
    metadata_path = esp_dir / 'metadata.json'

    joblib.dump(esp_artifacts[esp_id]['model'], model_path)

    preproc_bundle = {
        'median_width': median_width,
        'global_min': float(esp_artifacts[esp_id]['global_min']),
        'global_max': float(esp_artifacts[esp_id]['global_max']),
        'feature_cols': feature_cols,
        'feature_min': esp_artifacts[esp_id]['feature_min'],
        'feature_max': esp_artifacts[esp_id]['feature_max'],
        'scaler_pca': esp_artifacts[esp_id]['scaler_pca'],
        'pca': esp_artifacts[esp_id]['pca'],
        'k_95': int(esp_artifacts[esp_id]['k_95']),
        'model_type': 'sklearn',
        'esp_id': esp_id,
    }
    joblib.dump(preproc_bundle, preprocessing_path)

    metadata = {
        'source_notebook': 'each_esp_has_model_majority_vote_metrics.ipynb',
        'esp_id': esp_id,
        'model': 'SVM',
        'best_params_from_classic_ml_train': {'C': 5.0, 'gamma': 'auto', 'kernel': 'rbf'},
        'split': 'group train/test split by test_N folders, test_size=0.2, random_state=42',
        'model_path': str(model_path),
        'preprocessing_path': str(preprocessing_path),
    }
    metadata_path.write_text(json.dumps(metadata, indent=2), encoding='utf-8')

    esp_artifacts[esp_id].update({
        'model_path': model_path,
        'preprocessing_path': preprocessing_path,
        'metadata_path': metadata_path,
    })

    print(f'[{esp_id}] model_path: {model_path}')
    print(f'[{esp_id}] preprocessing_path: {preprocessing_path}')
    print(f'[{esp_id}] metadata_path: {metadata_path}')

[dev1] model_path: /home/gleb/learning/CSI-activity-detection/artifacts/classic_ml_majority_vote_metrics/dev1/svm_model.joblib
[dev1] preprocessing_path: /home/gleb/learning/CSI-activity-detection/artifacts/classic_ml_majority_vote_metrics/dev1/preprocessing.joblib
[dev1] metadata_path: /home/gleb/learning/CSI-activity-detection/artifacts/classic_ml_majority_vote_metrics/dev1/metadata.json
[dev2] model_path: /home/gleb/learning/CSI-activity-detection/artifacts/classic_ml_majority_vote_metrics/dev2/svm_model.joblib
[dev2] preprocessing_path: /home/gleb/learning/CSI-activity-detection/artifacts/classic_ml_majority_vote_metrics/dev2/preprocessing.joblib
[dev2] metadata_path: /home/gleb/learning/CSI-activity-detection/artifacts/classic_ml_majority_vote_metrics/dev2/metadata.json
[dev3] model_path: /home/gleb/learning/CSI-activity-detection/artifacts/classic_ml_majority_vote_metrics/dev3/svm_model.joblib
[dev3] preprocessing_path: /home/gleb/learning/CSI-activity-detection/artifacts/classic

## Majority vote across 3 ESP models

Для каждого `test_N` каждая ESP-модель даёт свой итоговый класс (внутри своей ESP сначала majority vote по её `.data` файлам).

Затем финальная метка `test_N` выбирается majority vote по трём ответам (`dev1`, `dev2`, `dev3`).

In [11]:
def true_class_from_label(label_name: str) -> int:
    return 0 if label_name == 'label_00' else 1


esp_group_vote_rows = []

for esp_id in esp_ids:
    split = esp_splits[esp_id]
    model_path = esp_artifacts[esp_id]['model_path']
    preprocessing_path = esp_artifacts[esp_id]['preprocessing_path']

    test_groups_sorted = sorted(split['test_groups'])

    for idx, test_group in enumerate(test_groups_sorted, start=1):
        data_files = sorted([fp for fp in split['test_files'] if get_test_group_dir(fp) == test_group])
        if not data_files:
            continue

        file_results = [run_inference(model_path, preprocessing_path, data_file) for data_file in data_files]
        file_predictions = [int(result['predicted_class']) for result in file_results]

        votes_static = file_predictions.count(0)
        votes_dynamic = file_predictions.count(1)
        esp_pred = 1 if votes_dynamic >= votes_static else 0

        group_label = parse_path_meta(test_group)['label']
        y_true = true_class_from_label(group_label)

        esp_group_vote_rows.append({
            'group_path': str(test_group),
            'test_id': test_group.name,
            'label': group_label,
            'y_true': y_true,
            'esp_id': esp_id,
            'n_files': len(data_files),
            'esp_votes_static': votes_static,
            'esp_votes_dynamic': votes_dynamic,
            'esp_pred': esp_pred,
        })

        if idx % 50 == 0:
            print(f'[{esp_id}] processed test groups: {idx}/{len(test_groups_sorted)}')

esp_group_preds_df = pd.DataFrame(esp_group_vote_rows)

group_rows = []
for group_path, group_df in esp_group_preds_df.groupby('group_path', sort=True):
    preds = [int(x) for x in group_df['esp_pred'].tolist()]
    n_esp_votes = len(preds)

    if n_esp_votes < len(esp_ids):
        continue

    votes_static = preds.count(0)
    votes_dynamic = preds.count(1)
    y_pred_final = 1 if votes_dynamic >= votes_static else 0

    first = group_df.iloc[0]
    group_rows.append({
        'group_path': group_path,
        'test_id': first['test_id'],
        'label': first['label'],
        'y_true': int(first['y_true']),
        'n_esp_votes': n_esp_votes,
        'final_votes_static': votes_static,
        'final_votes_dynamic': votes_dynamic,
        'y_pred_final': y_pred_final,
    })

final_vote_results_df = pd.DataFrame(group_rows).sort_values('group_path').reset_index(drop=True)

print(f'ESP-level group votes: {len(esp_group_preds_df)} rows')
print(f'Final groups scored with 3-model majority vote: {len(final_vote_results_df)}')
final_vote_results_df.head()

[dev1] processed test groups: 50/1600
[dev1] processed test groups: 100/1600
[dev1] processed test groups: 150/1600
[dev1] processed test groups: 200/1600
[dev1] processed test groups: 250/1600
[dev1] processed test groups: 300/1600
[dev1] processed test groups: 350/1600
[dev1] processed test groups: 400/1600
[dev1] processed test groups: 450/1600
[dev1] processed test groups: 500/1600
[dev1] processed test groups: 550/1600
[dev1] processed test groups: 600/1600
[dev1] processed test groups: 650/1600
[dev1] processed test groups: 700/1600
[dev1] processed test groups: 750/1600
[dev1] processed test groups: 800/1600
[dev1] processed test groups: 850/1600
[dev1] processed test groups: 900/1600
[dev1] processed test groups: 950/1600
[dev1] processed test groups: 1000/1600
[dev1] processed test groups: 1050/1600
[dev1] processed test groups: 1100/1600
[dev1] processed test groups: 1150/1600
[dev1] processed test groups: 1200/1600
[dev1] processed test groups: 1250/1600
[dev1] processed tes

,group_path,test_id,label,y_true,n_esp_votes,final_votes_static,final_votes_dynamic,y_pred_final
0,/home/gleb/learning/CSI-activity-detection/wif...,test_01_2,label_00,0,3,3,0,0
1,/home/gleb/learning/CSI-activity-detection/wif...,test_02_4,label_00,0,3,0,3,1
2,/home/gleb/learning/CSI-activity-detection/wif...,test_03_4,label_00,0,3,2,1,0
3,/home/gleb/learning/CSI-activity-detection/wif...,test_03_5,label_00,0,3,2,1,0
4,/home/gleb/learning/CSI-activity-detection/wif...,test_05_1,label_00,0,3,3,0,0


## Test metrics after final cross-ESP majority vote

In [14]:
y_test_group = final_vote_results_df['y_true'].to_numpy()
y_pred_group = final_vote_results_df['y_pred_final'].to_numpy()

precision, recall, f1, support = precision_recall_fscore_support(
    y_test_group,
    y_pred_group,
    labels=[0, 1],
    zero_division=0,
)

cm = confusion_matrix(y_test_group, y_pred_group, labels=[0, 1])
acc = accuracy_score(y_test_group, y_pred_group)

final_metrics_df = pd.DataFrame({
    'class': ['static / no motion', 'dynamic / motion'],
    'precision': precision,
    'recall': recall,
    'f1_score': f1,
    'support': support,
})

final_summary_df = pd.DataFrame([{
    'accuracy': acc,
    'n_test_groups': int(len(final_vote_results_df)),
    'n_models_in_vote': int(len(esp_ids)),
}])

print('Confusion matrix (rows=true, cols=pred):')
print(cm)
print()
print(classification_report(
    y_test_group,
    y_pred_group,
    labels=[0, 1],
    target_names=['static / no motion', 'dynamic / motion'],
    zero_division=0,
))
print()
print('Final majority-vote summary:')
final_summary_df

Confusion matrix (rows=true, cols=pred):
[[ 331   69]
 [  35 1165]]

                    precision    recall  f1-score   support

static / no motion       0.90      0.83      0.86       400
  dynamic / motion       0.94      0.97      0.96      1200

          accuracy                           0.94      1600
         macro avg       0.92      0.90      0.91      1600
      weighted avg       0.93      0.94      0.93      1600


Final majority-vote summary:


,accuracy,n_test_groups,n_models_in_vote
0,0.935,1600,3


## Test errors after final cross-ESP majority vote

In [15]:
final_errors_df = final_vote_results_df[final_vote_results_df['y_true'] != final_vote_results_df['y_pred_final']].copy()

print(f'Final majority-vote test group errors: {len(final_errors_df)}')
final_errors_df

Final majority-vote test group errors: 104


,group_path,test_id,label,y_true,n_esp_votes,final_votes_static,final_votes_dynamic,y_pred_final
1,/home/gleb/learning/CSI-activity-detection/wif...,test_02_4,label_00,0,3,0,3,1
17,/home/gleb/learning/CSI-activity-detection/wif...,test_22_3,label_00,0,3,1,2,1
20,/home/gleb/learning/CSI-activity-detection/wif...,test_26_1,label_00,0,3,1,2,1
22,/home/gleb/learning/CSI-activity-detection/wif...,test_28_4,label_00,0,3,1,2,1
24,/home/gleb/learning/CSI-activity-detection/wif...,test_30_1,label_00,0,3,1,2,1
...,...,...,...,...,...,...,...,...
1589,/home/gleb/learning/CSI-activity-detection/wif...,test_90_2,label_03,1,3,2,1,0
1591,/home/gleb/learning/CSI-activity-detection/wif...,test_91_2,label_03,1,3,2,1,0
1594,/home/gleb/learning/CSI-activity-detection/wif...,test_92_5,label_03,1,3,2,1,0
1597,/home/gleb/learning/CSI-activity-detection/wif...,test_97_1,label_03,1,3,2,1,0
